**Shows how the T-Test for cosine distances was obtained**


In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas matplotlib
from bertopic import BERTopic
import pandas as pd

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

data = pd.read_csv('highconf_label_data.csv')

cat = 2.0

data = data[data['label'] == cat]
data = data.dropna(subset=['body']).reset_index(drop=True)

data = data['body'].to_list()

GPU Available: True


In [ ]:
from hdbscan import HDBSCAN

hdbscan_model = HDBSCAN(min_cluster_size=20,  # min cluster size before it can become its own
                        min_samples=15,       # how many neighbors it has before its part of a clsuter
                        cluster_selection_epsilon=0.1,
                        metric="euclidean",
                        prediction_data=True)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(ngram_range=(1, 2), stop_words="english")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
topic_model = BERTopic(embedding_model=embedding_model,
                       hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer_model,
                       verbose=True)
topics, probs = topic_model.fit_transform(data)

In [ ]:
topic_model.get_topic_info()

In [ ]:
pl_embed = topic_model.topic_embeddings_


In [ ]:
data = pd.read_csv('highconf_label_data.csv')

cat = 1.0

data = data[data['label'] == cat]
data = data.dropna(subset=['body']).reset_index(drop=True)

data = data['body'].to_list()

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(ngram_range=(1, 2), stop_words="english")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
topic_model = BERTopic(embedding_model=embedding_model,
                       hdbscan_model=hdbscan_model,
                       vectorizer_model=vectorizer_model,
                       verbose=True)
topics, probs = topic_model.fit_transform(data)

In [ ]:
topic_model.get_topic_info()

In [ ]:
pc_embed = topic_model.topic_embeddings_

In [ ]:
import numpy as np
from scipy import stats
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity_matrix = cosine_similarity(pl_embed)

distance_matrix = 1 - cosine_similarity(pl_embed)

from scipy.spatial.distance import squareform

pl_distances = squareform(distance_matrix, checks=False)

pl_distances

array([0.2624855 , 0.1350761 , 0.15209234, ..., 0.6033461 , 0.7009525 ,
       0.8185123 ], dtype=float32)

In [ ]:
similarity_matrixPc = cosine_similarity(pc_embed)

distance_matrixPc = 1 - cosine_similarity(pc_embed)

from scipy.spatial.distance import squareform

pc_distances = squareform(distance_matrixPc, checks=False)

In [ ]:
t_stat, p_val = stats.ttest_ind(pc_distances, pl_distances)

print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.8f}")

T-statistic: 4.9378, P-value: 0.00000082


### T-Test Results

- **First run**  
  - *T-statistic:* 2.17  
  - *P-value:* 0.0297  

- **Second run**  
  - *T-statistic:* 8.11  
  - *P-value:* < 0.00000001  

- **Third run**  
  - *T-statistic:* 4.94  
  - *P-value:* 0.00000082  


